In [2]:
# imports
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from scipy.spatial.distance import cdist
import osmnx as ox
import warnings
warnings.filterwarnings('ignore')
import gtfs_kit as gk
import polars as pl
import geopandas as gpd
from multiprocessing import Pool, cpu_count
from functools import partial
from tqdm import tqdm
pd.set_option('display.max_columns', None)

# constants
CBD_COORDS = np.array([[-1.2864, 36.8172]])
PEAK_HOURS = [6, 7, 8, 9, 17, 18, 19, 20]

In [3]:
# load data
print("Loading data...")
# load gtfs data
print("Loading GTFS data...")
feed_path = '/home/dataopske/Desktop/jav/data/raw/digitalmatatu/GTFS_FEED_2019.zip'
feed = gk.read_feed(feed_path, dist_units='km')
gtfs_routes = feed.routes
gtfs_trips = feed.trips
gtfs_stop_times = feed.stop_times
gtfs_stops = feed.stops
gtfs_shapes = feed.shapes
print(f"✓ GTFS loaded: {len(gtfs_routes)} routes, {len(gtfs_stops)} stops")
print(f" Route IDs: {gtfs_routes['route_id'].unique()[:5]}...")

Loading data...
Loading GTFS data...
✓ GTFS loaded: 136 routes, 4284 stops
 Route IDs: <StringArray>
['10000107D11', '10000114011', '10000116011', '10100011A11', '10200010811']
Length: 5, dtype: string...


In [4]:
# load ward lookup early for gtfs_stops
ward_lookup = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/stop_ward_lookup.csv')
gtfs_stops = gtfs_stops.merge(ward_lookup[['stop_id', 'ward']], on='stop_id', how='left')
gtfs_stops['ward'] = gtfs_stops['ward'].fillna('unknown')
print(f"✓ Wards merged to GTFS stops: {gtfs_stops['ward'].nunique()} unique wards")

# load the daily aggregated one:
model2_daily = pd.read_parquet('/home/dataopske/Desktop/jav/data/processed/model2_traffic_daily.parquet')
df1 = gpd.read_parquet('/home/dataopske/Desktop/jav/data/processed/spatio_temporal_calendar.parquet')
print("✓ Processed data (traffic, spatio_temporal) loaded.")

# load osm road network
print("Downloading road network...")
try:
    G = ox.load_graphml('/home/dataopske/Desktop/jav/data/processed/nairobi_drive.graphml')
except:
    G = ox.graph_from_place("Nairobi, Kenya", network_type='drive')
    ox.save_graphml(G, 'nairobi_roads.graphml')
nodes_gdf = ox.graph_to_gdfs(G, edges=False)
print("✓ Road network loaded.")

✓ Wards merged to GTFS stops: 79 unique wards
✓ Processed data (traffic, spatio_temporal) loaded.
✓ Road network loaded.


In [5]:
# helper functions
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371
    phi1, phi2 = np.radians(lat1), np.radians(lat2)
    dphi = np.radians(lat2 - lat1)
    dlambda = np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(phi1) * np.cos(phi2) * np.sin(dlambda/2)**2
    return 2 * R * np.arcsin(np.sqrt(a))

def get_road_features(lat, lon, G, nodes_gdf):
    try:
        nearest_node = ox.distance.nearest_nodes(G, lon, lat)
        degree = G.degree(nearest_node)
        is_intersection = degree >= 3
       
        # get road type from edges
        edges = list(G.edges(nearest_node, data=True))
        if edges:
            road_type = edges[0][2].get('highway', 'unknown')
            if isinstance(road_type, list):
                road_type = road_type[0]
        else:
            road_type = 'unknown'
       
        # distance to major road
        major_edges = [(u, v, d) for u, v, d in G.edges(data=True)
                       if d.get('highway') in ['primary', 'secondary', 'trunk']]
        if major_edges:
            major_nodes = set([u for u, v, d in major_edges] + [v for u, v, d in major_edges])
            major_coords = nodes_gdf.loc[list(major_nodes), ['y', 'x']].values
            dists = cdist([[lat, lon]], major_coords)[0]
            dist_to_major = dists.min() * 111000 # to meters
        else:
            dist_to_major = np.nan
           
        return {
            'nearest_node_degree': degree,
            'is_intersection': is_intersection,
            'road_type': road_type,
            'distance_to_major_road': dist_to_major
        }
    except:
        return {
            'nearest_node_degree': np.nan,
            'is_intersection': False,
            'road_type': 'unknown',
            'distance_to_major_road': np.nan
        }

def get_stop_spacing_features(lat, lon, stop_tree, k=4):
    distances, indices = stop_tree.query([lat, lon], k=k)
   
    # if this is an existing stop, distances[0] = 0, so use [1:4]
    # if candidate, use [0:3]
    if distances[0] < 0.0001: # existing stop
        relevant_dists = distances[1:4] * 111000 # to meters
    else: # candidate
        relevant_dists = distances[0:3] * 111000
   
    return {
        'distance_to_nearest_stop': relevant_dists[0] if len(relevant_dists) > 0 else np.nan,
        'distance_to_2nd_nearest': relevant_dists[1] if len(relevant_dists) > 1 else np.nan,
        'distance_to_3rd_nearest': relevant_dists[2] if len(relevant_dists) > 2 else np.nan,
        'avg_spacing_3_nearest': np.mean(relevant_dists),
        'spacing_regularity': np.std(relevant_dists)
    }

def get_stops_in_radius(lat, lon, stop_tree, stop_coords, radius_km):
    indices = stop_tree.query_ball_point([lat, lon], radius_km / 111)
    return len(indices)

def get_population_features(lat, lon, ward, df1_agg):
    ward_data = df1_agg[df1_agg['ward'] == ward]
    if len(ward_data) == 0:
        return {
            'pop_within_500m': 0,
            'pop_within_1km': 0,
            'pop_density_500m': 0,
            'pop_not_served_nearby': 0,
            'poverty_rate_weighted_pop': 0
        }
   
    ward_data = ward_data.iloc[0]
    pop_density = ward_data['pop_density']
    poverty_rate = ward_data['poverty_rate']
   
    area_500m = np.pi * 0.5**2
    area_1km = np.pi * 1**2
   
    pop_500m = pop_density * area_500m
    pop_1km = pop_density * area_1km
   
    return {
        'pop_within_500m': int(pop_500m),
        'pop_within_1km': int(pop_1km),
        'pop_density_500m': pop_density,
        'pop_not_served_nearby': int(ward_data.get('pop_not_served', 0) * 0.3), # approximate
        'poverty_rate_weighted_pop': pop_500m * (poverty_rate / 100)
    }

# SPEEDUP: Precompute service features for all existing stops once (vectorized)
def precompute_service_features(gtfs_stop_times, gtfs_trips, PEAK_HOURS):
    print("Precomputing service features for all stops...")
    service_dict = {}
    stop_times = gtfs_stop_times.copy()
    stop_times['hour'] = pd.to_datetime(stop_times['arrival_time'], format='%H:%M:%S', errors='coerce').dt.hour
    stop_times = stop_times.dropna(subset=['hour'])
    
    for stop_id in tqdm(stop_times['stop_id'].unique(), desc="Precomputing stops"):
        st = stop_times[stop_times['stop_id'] == stop_id]
        if len(st) == 0:
            service_dict[stop_id] = {
                'route_count_serving': 0,
                'trips_per_day': 0,
                'trips_per_hour_peak': 0,
                'trips_per_hour_offpeak': 0,
                'avg_headway_minutes': np.nan,
                'service_span_hours': 0,
                'routes_within_500m': 0
            }
            continue
        
        trips_at_stop = st['trip_id'].unique()
        route_count = gtfs_trips[gtfs_trips['trip_id'].isin(trips_at_stop)]['route_id'].nunique()
        trips_per_day = len(st)
        
        peak_trips = st[st['hour'].isin(PEAK_HOURS)]
        offpeak_trips = st[~st['hour'].isin(PEAK_HOURS)]
        
        trips_per_hour_peak = len(peak_trips) / len(PEAK_HOURS) if len(PEAK_HOURS) > 0 else 0
        trips_per_hour_offpeak = len(offpeak_trips) / (24 - len(PEAK_HOURS)) if len(offpeak_trips) > 0 else 0
        
        avg_headway = (24 * 60) / trips_per_day if trips_per_day > 0 else np.nan
        service_span = st['hour'].max() - st['hour'].min() if len(st) > 0 else 0
        
        service_dict[stop_id] = {
            'route_count_serving': route_count,
            'trips_per_day': trips_per_day,
            'trips_per_hour_peak': trips_per_hour_peak,
            'trips_per_hour_offpeak': trips_per_hour_offpeak,
            'avg_headway_minutes': avg_headway,
            'service_span_hours': service_span,
            'routes_within_500m': route_count  # approximation
        }
    print("✓ Service features precomputed.")
    return service_dict

def get_service_features(stop_id, service_dict, gtfs_stop_times, gtfs_trips, all_stops_gdf, lat, lon):
    if pd.isna(stop_id) or stop_id not in service_dict:
        # candidate location - check nearby routes
        nearby = all_stops_gdf[
            np.sqrt((all_stops_gdf['stop_lat'] - lat)**2 +
                   (all_stops_gdf['stop_lon'] - lon)**2) < 0.005
        ]
        if len(nearby) > 0:
            nearby_stop_ids = nearby['stop_id'].values
            nearby_stop_times = gtfs_stop_times[gtfs_stop_times['stop_id'].isin(nearby_stop_ids)]
            nearby_trips = nearby_stop_times['trip_id'].unique()
            routes_within_500m = gtfs_trips[gtfs_trips['trip_id'].isin(nearby_trips)]['route_id'].nunique()
        else:
            routes_within_500m = 0
       
        return {
            'route_count_serving': 0,
            'trips_per_day': 0,
            'trips_per_hour_peak': 0,
            'trips_per_hour_offpeak': 0,
            'avg_headway_minutes': np.nan,
            'service_span_hours': 0,
            'routes_within_500m': routes_within_500m
        }
   
    return service_dict[stop_id]

# Update get_traffic_features to use daily lookup (simpler, no hourly filter)
def get_traffic_features(lat, lon, model2_daily, cell_daily_tree, cell_daily_ids):
    # Find nearest cell
    try:
        distances, indices = cell_daily_tree.query([lat, lon], k=1)
        # Handle scalar return for k=1 (scipy quirk)
        if np.isscalar(indices):
            nearest_cell_idx = int(indices)
        else:
            nearest_cell_idx = int(indices[0])
        nearest_cell = cell_daily_ids[nearest_cell_idx]
    except Exception as e:
        print(f"Query failed for ({lat}, {lon}): {e}")  # Optional: Remove after debug
        return {k: np.nan for k in [
            'avg_speed_daily', 'avg_speed_peak', 'avg_speed_offpeak',
            'congestion_pct_daily', 'congestion_pct_peak',
            'trip_count_daily', 'trip_count_peak',
            'demand_variability_cv', 'dominant_congestion_level'
        ]}
   
    # Direct lookup - no hourly agg needed
    cell_rows = model2_daily[model2_daily['cell_id'] == nearest_cell]
    if len(cell_rows) == 0:
        return {k: np.nan for k in [
            'avg_speed_daily', 'avg_speed_peak', 'avg_speed_offpeak',
            'congestion_pct_daily', 'congestion_pct_peak',
            'trip_count_daily', 'trip_count_peak',
            'demand_variability_cv', 'dominant_congestion_level'
        ]}
    
    cell_row = cell_rows.iloc[0]
   
    return {
        'avg_speed_daily': cell_row['avg_speed_daily'],
        'avg_speed_peak': cell_row['avg_speed_peak'],
        'avg_speed_offpeak': cell_row['avg_speed_offpeak'],
        'congestion_pct_daily': cell_row['congestion_pct_daily'],
        'congestion_pct_peak': cell_row['congestion_pct_peak'],
        'trip_count_daily': cell_row['trip_count_daily'],
        'trip_count_peak': cell_row['trip_count_peak'],
        'demand_variability_cv': cell_row['demand_variability_cv'],
        'dominant_congestion_level': cell_row['dominant_congestion_level']
    }

def get_ward_features(ward, ward_features_df):
    ward_data = ward_features_df[ward_features_df['ward'] == ward]
    if len(ward_data) == 0:
        return {
            'ward_pct_access': 0,
            'ward_population': 0,
            'ward_pop_density': 0,
            'ward_poverty_rate': 0,
            'ward_pop_not_served': 0,
            'is_benchmark_ward': False,
            'ward_service_per_capita': 0,
            'ward_category': 'unknown'
        }
   
    ward_data = ward_data.iloc[0]
    return {
        'ward_pct_access': ward_data.get('pct_access', 0),
        'ward_population': ward_data.get('population', 0),
        'ward_pop_density': ward_data.get('pop_density', 0),
        'ward_poverty_rate': ward_data.get('poverty_rate', 0),
        'ward_pop_not_served': ward_data.get('pop_not_served', 0),
        'is_benchmark_ward': ward_data.get('is_benchmark', False),
        'ward_service_per_capita': ward_data.get('service_per_capita', 0),
        'ward_category': ward_data.get('ward_category', 'unknown')
    }

def get_spatial_features(lat, lon):
    distance_to_cbd = haversine_km(lat, lon, CBD_COORDS[0][0], CBD_COORDS[0][1])
    return {
        'distance_to_cbd': distance_to_cbd
    }

def get_derived_features(features):
    stops_1km = features.get('stops_within_1km', 1)
    ward_access = features.get('ward_pct_access', 1)
    trips_per_day = features.get('trips_per_day', 0)
    pop_500m = features.get('pop_within_500m', 1)
    pop_not_served = features.get('pop_not_served_nearby', 0)
    poverty_weighted = features.get('poverty_rate_weighted_pop', 0)
    degree = features.get('nearest_node_degree', 1)
    dist_major = features.get('distance_to_major_road', 1)
   
    return {
        'coverage_efficiency_nearby': ward_access / (stops_1km + 1),
        'demand_supply_ratio': pop_500m / (trips_per_day + 1),
        'network_accessibility': degree / (dist_major + 1),
        'equity_score': pop_not_served * poverty_weighted
    }

def extract_stop_features(stop_data, G, nodes_gdf, stop_tree, stop_coords,
                          df1_agg, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops,
                          model2_daily, cell_daily_tree, cell_daily_ids, ward_features):  # Updated params
    idx, stop = stop_data
   
    lat, lon = stop['stop_lat'], stop['stop_lon']
    stop_id = stop['stop_id']
    ward = stop.get('ward', 'unknown')
   
    features = {
        'stop_id': stop_id,
        'lat': lat,
        'lon': lon,
        'ward': ward,
        'is_existing_stop': True
    }
   
    features.update(get_road_features(lat, lon, G, nodes_gdf))
    features.update(get_stop_spacing_features(lat, lon, stop_tree))
   
    stops_500m = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 0.5)
    stops_1km = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 1.0)
    features['stops_within_500m'] = stops_500m
    features['stops_within_1km'] = stops_1km
    features['stop_density_1km'] = stops_1km / (np.pi * 1**2)
   
    features.update(get_population_features(lat, lon, ward, df1_agg))
    features.update(get_service_features(stop_id, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops, lat, lon))  # SPEEDUP: Uses precomputed dict
    features.update(get_traffic_features(lat, lon, model2_daily, cell_daily_tree, cell_daily_ids))  # Updated
    features.update(get_ward_features(ward, ward_features))
    features.update(get_spatial_features(lat, lon))
    features.update(get_derived_features(features))
   
    features['is_good_stop'] = 1 if features['is_benchmark_ward'] else 0
   
    return features

# SPEEDUP: Function to generate candidates for a single ward (for parallel negatives)
def generate_ward_candidates(ward_data, n_negatives_per_ward, G, nodes_gdf, stop_tree, stop_coords,
                             df1_agg, gtfs_stop_times, gtfs_trips, gtfs_stops, service_dict,
                             model2_traffic, cell_tree, cell_ids, ward_features):
    ward, ward_stops = ward_data
    if len(ward_stops) == 0:
        return []
    
    lats = ward_stops['lat'].values
    lons = ward_stops['lon'].values
    
    lat_min, lat_max = lats.min() - 0.01, lats.max() + 0.01
    lon_min, lon_max = lons.min() - 0.01, lons.max() + 0.01
    
    candidates = []
    attempts = 0  # To avoid infinite loop if hard to place
    while len(candidates) < n_negatives_per_ward and attempts < n_negatives_per_ward * 10:
        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)
        
        # check not too close to existing stop
        dists, _ = stop_tree.query([lat, lon], k=1)
        dist_m = float(dists) * 111000 if isinstance(dists, (int, float)) else dists[0] * 111000
        
        if dist_m < 300:
            attempts += 1
            continue
        
        features = {
            'stop_id': f'CANDIDATE_{ward}_{len(candidates)}',
            'lat': lat,
            'lon': lon,
            'ward': ward,
            'is_existing_stop': False
        }
        
        features.update(get_road_features(lat, lon, G, nodes_gdf))
        features.update(get_stop_spacing_features(lat, lon, stop_tree))
        
        stops_500m = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 0.5)
        stops_1km = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 1.0)
        features['stops_within_500m'] = stops_500m
        features['stops_within_1km'] = stops_1km
        features['stop_density_1km'] = stops_1km / (np.pi * 1**2)
        
        features.update(get_population_features(lat, lon, ward, df1_agg))
        features.update(get_service_features(None, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops, lat, lon))
        features.update(get_traffic_features(lat, lon, model2_traffic, cell_tree, cell_ids))
        features.update(get_ward_features(ward, ward_features))
        features.update(get_spatial_features(lat, lon))
        features.update(get_derived_features(features))
        
        features['is_good_stop'] = 0
        
        candidates.append(features)
        attempts += 1
    
    return candidates

In [6]:
# build spatial indices
print("Building spatial indices...")
stop_coords = gtfs_stops[['stop_lat', 'stop_lon']].values
stop_tree = cKDTree(stop_coords)

# KDTree for traffic cells (daily version: lat first for query)
cell_daily_coords = model2_daily[['lat', 'lon']].values
cell_daily_tree = cKDTree(cell_daily_coords)
cell_daily_ids = model2_daily['cell_id'].values

# SPEEDUP: Use Polars for ward aggs
df1_pl = pl.from_pandas(df1.drop(columns=df1.select_dtypes(include=['geometry'])))  # Drop geo for agg
df1_agg_pl = df1_pl.group_by('ward').agg([
    pl.col('population').first(),
    pl.col('pop_density').first(),
    pl.col('poverty_rate').first(),
    pl.col('pop_not_served').first()
]).to_pandas()  # Back to pandas for compatibility
df1_agg = df1_agg_pl
print("✓ Spatial indices built.")

Building spatial indices...
✓ Spatial indices built.


In [7]:
# prepare ward features (Polars)
print("Preparing ward features...")
ward_features_pl = df1_pl.group_by('ward').agg([
    pl.col('population').first().alias('population'),
    pl.col('pop_density').first().alias('pop_density'),
    pl.col('pct_access').first().alias('pct_access'),
    pl.col('pop_served').first().alias('pop_served'),
    pl.col('pop_not_served').first().alias('pop_not_served'),
    pl.col('coverage_ratio').first().alias('coverage_ratio'),
    pl.col('poverty_rate').first().alias('poverty_rate'),
    pl.col('subcounty_gini').first().alias('subcounty_gini'),
    pl.col('trips_per_hour').mean().alias('trips_per_hour_mean'),
    pl.col('trips_per_hour').std().alias('trips_per_hour_std'),
    pl.col('trips_per_hour').sum().alias('trips_per_hour_sum'),
    pl.col('trips_per_1k_pop_per_hour').mean().alias('service_per_capita'),
    pl.col('trips_per_1k_pop_per_hour').std().alias('trips_per_1k_pop_per_hour_std'),
    pl.col('service_rank_overall').first().alias('service_rank_overall')
])
ward_features = ward_features_pl.to_pandas()
ward_features['is_benchmark'] = ward_features['pct_access'] >= 70
ward_features['ward_category'] = pd.cut(
    ward_features['pct_access'],
    bins=[0, 50, 70, 90, 100],
    labels=['severely_underserved', 'underserved', 'adequately_served', 'well_served']
)
print("✓ Ward features prepared.")

Preparing ward features...
✓ Ward features prepared.


In [10]:
# Testing mode
TEST_MODE = True  # Set to False for full run
N_TEST_STOPS = 100
N_TEST_NEG_PER_WARD = 10  # Small for speed; adjust as needed

In [11]:
# SPEEDUP: Precompute service (full or test subset)
print("Precomputing service features...")
if TEST_MODE:
    # Test mode: Precompute only on unique stop_ids from test stops (faster)
    test_stop_ids = gtfs_stops['stop_id'].unique()[:N_TEST_STOPS]  # Subset
    service_dict = precompute_service_features(gtfs_stop_times[gtfs_stop_times['stop_id'].isin(test_stop_ids)], gtfs_trips, PEAK_HOURS)
else:
    service_dict = precompute_service_features(gtfs_stop_times, gtfs_trips, PEAK_HOURS)
print("✓ Service features precomputed.")

Precomputing service features...
Precomputing service features for all stops...


Precomputing stops:  15%|█▍        | 13/89 [00:00<00:00, 123.75it/s]

Precomputing stops: 100%|██████████| 89/89 [00:00<00:00, 214.46it/s]

✓ Service features precomputed.
✓ Service features precomputed.


In [12]:
# extract features for existing stops (parallelized)
print("Extracting features for existing stops...")
# prepare data
stop_data_list = list(gtfs_stops.iterrows())
if TEST_MODE:
    stop_data_list = stop_data_list[:N_TEST_STOPS]  # Subset to first N_TEST_STOPS
    print(f"TEST MODE: Using only {len(stop_data_list)} stops")
extract_func = partial(
    extract_stop_features,
    G=G,
    nodes_gdf=nodes_gdf,
    stop_tree=stop_tree,
    stop_coords=stop_coords,
    df1_agg=df1_agg,
    service_dict=service_dict,
    gtfs_stop_times=gtfs_stop_times,
    gtfs_trips=gtfs_trips,
    gtfs_stops=gtfs_stops,
    model2_daily=model2_daily,
    cell_daily_tree=cell_daily_tree,
    cell_daily_ids=cell_daily_ids,
    ward_features=ward_features
)
n_cores = cpu_count() - 1
print(f"Using {n_cores} cores for {len(stop_data_list)} stops...")
with Pool(n_cores) as pool:
    all_features = list(tqdm(
        pool.imap(extract_func, stop_data_list, chunksize=20),
        total=len(stop_data_list),
        desc="Processing stops"
    ))
stops_df = pd.DataFrame(all_features)
print(f"Processed {len(stops_df)} stops")

Extracting features for existing stops...
TEST MODE: Using only 100 stops
Using 7 cores for 100 stops...


Processing stops: 100%|██████████| 100/100 [00:44<00:00,  2.23it/s]


Processed 100 stops


In [13]:
stops_df.isna().sum()  # Check for NaNs

stop_id                        0
lat                            0
lon                            0
ward                           0
is_existing_stop               0
nearest_node_degree            0
is_intersection                0
road_type                      0
distance_to_major_road         0
distance_to_nearest_stop       0
distance_to_2nd_nearest        0
distance_to_3rd_nearest        0
avg_spacing_3_nearest          0
spacing_regularity             0
stops_within_500m              0
stops_within_1km               0
stop_density_1km               0
pop_within_500m                0
pop_within_1km                 0
pop_density_500m               0
pop_not_served_nearby          0
poverty_rate_weighted_pop      0
route_count_serving            0
trips_per_day                  0
trips_per_hour_peak            0
trips_per_hour_offpeak         0
avg_headway_minutes           11
service_span_hours             0
routes_within_500m             0
avg_speed_daily                0
avg_speed_

In [14]:
stops_df['ward_category'].value_counts()

ward_category
underserved             69
well_served             20
severely_underserved    11
Name: count, dtype: int64

In [15]:
# load and merge additional data
pop_df = pd.read_csv('/home/dataopske/Desktop/jav/data/processed/stop_population_buffers.csv')
# drop old pop columns first
stops_df = stops_df.drop(columns=['pop_within_200m', 'pop_within_500m', 'pop_within_1km'], errors='ignore')
# merge new population data
stops_df = stops_df.merge(pop_df, on='stop_id', how='left')

In [16]:
# SPEEDUP: Vectorized recompute derived (no apply)
print("Recomputing derived features with real pop data...")
# Vector ops for derived (using merged pop columns directly)
stops_df['coverage_efficiency_nearby'] = stops_df['ward_pct_access'] / (stops_df['stops_within_1km'] + 1)
stops_df['demand_supply_ratio'] = stops_df['pop_within_500m'] / (stops_df['trips_per_day'] + 1)
stops_df['network_accessibility'] = stops_df['nearest_node_degree'] / (stops_df['distance_to_major_road'] + 1)
stops_df['equity_score'] = stops_df['pop_not_served_nearby'] * stops_df['poverty_rate_weighted_pop']
print("✓ Derived features recomputed with real pop data")
stops_df.head(2)

Recomputing derived features with real pop data...
✓ Derived features recomputed with real pop data


,stop_id,lat,lon,ward,is_existing_stop,nearest_node_degree,is_intersection,road_type,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,dominant_congestion_level,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,is_benchmark_ward,ward_service_per_capita,ward_category,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop,pop_within_200m,pop_within_500m,pop_within_1km
0,0001RLW,-1.290884,36.828242,Nairobi Central Ward,True,4,True,unclassified,185.404122,173.684507,231.853573,412.893917,272.810666,101.860683,9,57,18.143664,6034.16,0,947.843636,7,14,1.75,0,102.857143,1,7,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,1.324902,1.724138,372.666667,0.021459,0.0,1,799,5590,27759
1,0002KOJ,-1.281230,36.822596,Nairobi Central Ward,True,3,True,residential,20.568573,5.177885,21.286282,23.097604,16.520590,8.054520,14,66,21.008452,6034.16,0,947.843636,0,0,0.00,0,NaN,0,46,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.830851,1.492537,5161.000000,0.139091,0.0,1,757,5161,23044


In [17]:
stops_df.describe()

,lat,lon,nearest_node_degree,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,ward_service_per_capita,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop,pop_within_200m,pop_within_500m,pop_within_1km
count,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.00000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.0,89.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.000000,100.00000,100.000000,1.000000e+02,100.000000,100.000000,100.000000,100.000000
mean,-1.206013,36.795097,5.500000,148.652453,69.072054,179.469863,236.237926,161.593281,73.140493,12.14000,28.18000,8.969973,6399.719300,14640.030000,781.877849,1.28000,1.350000,0.168750,0.0,1228.391653,0.050000,6.30000,17.796361,11.694676,20.867242,58.150123,92.749086,409.41000,165.770000,0.446975,59.268918,108220.948170,6399.719300,17.224000,48801.860000,28.122083,11.095927,3.147176,2197.40900,0.289633,8.698627e+06,0.200000,463.850000,3048.660000,13129.570000
std,0.040742,0.065378,1.514242,200.116658,128.173733,203.068349,216.662040,175.143453,60.467233,8.27314,20.66392,6.577530,8635.474776,7809.266729,679.442774,1.07384,1.565893,0.195737,0.0,373.582363,0.219043,15.53003,0.262774,0.182647,0.336537,2.707299,0.585230,138.19479,55.353931,0.034105,21.263902,45212.935034,8635.474776,2.399205,26031.299221,39.455023,5.982415,2.742442,6250.69665,0.427013,5.705017e+06,0.402015,1093.860455,6698.618923,28427.817854
min,-1.290884,36.643126,2.000000,0.903475,0.730793,6.450994,20.499269,16.520590,2.494179,1.00000,2.00000,0.636620,3569.890000,0.000000,405.623790,0.00000,0.000000,0.000000,0.0,102.857143,0.000000,1.00000,16.977835,11.514725,20.060219,54.693287,90.625000,160.00000,66.000000,0.413343,30.372612,18997.268882,3569.890000,7.500000,0.000000,0.985044,0.830851,0.925926,0.00000,0.003235,0.000000e+00,0.000000,0.000000,0.000000,0.000000
25%,-1.219856,36.760790,5.000000,14.959869,11.106333,60.666202,109.451299,65.088357,30.567502,6.00000,16.00000,5.092958,3569.890000,11984.000000,501.877524,1.00000,1.000000,0.125000,0.0,720.000000,0.000000,1.00000,17.733884,11.535665,20.832993,56.990763,92.241379,276.00000,114.000000,0.413343,51.540508,66325.319937,3569.890000,17.900000,39949.000000,16.862692,8.022372,1.665649,0.00000,0.021163,4.860996e+06,0.000000,0.000000,0.000000,0.000000
50%,-1.202433,36.782746,6.000000,32.428583,23.248811,106.840447,152.143821,98.591540,54.560449,11.00000,22.50000,7.161972,3569.890000,19794.000000,501.877524,1.00000,1.000000,0.125000,0.0,1440.000000,0.000000,1.00000,17.733884,11.582853,20.832993,57.133067,92.968750,520.00000,208.000000,0.433371,51.540508,136158.568704,3569.890000,17.900000,65982.000000,16.862692,10.814450,2.240892,0.00000,0.141708,9.934164e+06,0.000000,0.000000,0.000000,1209.000000
75%,-1.183690,36.824112,6.000000,197.153080,58.761128,222.352602,275.396627,176.113762,102.415983,16.25000,40.50000,12.891550,6034.160000,19794.000000,947.843636,1.00000,1.000000,0.125000,0.0,1440.000000,0.000000,2.00000,18.047650,11.942590,21.100180,57.188680,93.181818,520.00000,208.000000,0.470559,58.694894,136158.568704,6034.160000,17.900000,65982.000000,16.862692,14.545204,3.031795,1619.50000,0.392214,9.934164e+06,0.0000

In [18]:
# generate negative examples (parallelized)
# SPEEDUP: Function to generate candidates for a single ward (for parallel negatives) - FULL DEFINITION
def generate_ward_candidates(ward_data, n_negatives_per_ward, G, nodes_gdf, stop_tree, stop_coords,
                             df1_agg, gtfs_stop_times, gtfs_trips, gtfs_stops, service_dict,
                             model2_daily, cell_daily_tree, cell_daily_ids, ward_features):
    ward, ward_stops = ward_data
    if len(ward_stops) == 0:
        return []
    
    lats = ward_stops['lat'].values
    lons = ward_stops['lon'].values
    
    lat_min, lat_max = lats.min() - 0.01, lats.max() + 0.01
    lon_min, lon_max = lons.min() - 0.01, lons.max() + 0.01
    
    candidates = []
    attempts = 0  # To avoid infinite loop if hard to place
    while len(candidates) < n_negatives_per_ward and attempts < n_negatives_per_ward * 10:
        lat = np.random.uniform(lat_min, lat_max)
        lon = np.random.uniform(lon_min, lon_max)
        
        # check not too close to existing stop
        dists, _ = stop_tree.query([lat, lon], k=1)
        dist_m = float(dists) * 111000 if isinstance(dists, (int, float)) else dists[0] * 111000
        
        if dist_m < 300:
            attempts += 1
            continue
        
        features = {
            'stop_id': f'CANDIDATE_{ward}_{len(candidates)}',
            'lat': lat,
            'lon': lon,
            'ward': ward,
            'is_existing_stop': False
        }
        
        features.update(get_road_features(lat, lon, G, nodes_gdf))
        features.update(get_stop_spacing_features(lat, lon, stop_tree))
        
        stops_500m = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 0.5)
        stops_1km = get_stops_in_radius(lat, lon, stop_tree, stop_coords, 1.0)
        features['stops_within_500m'] = stops_500m
        features['stops_within_1km'] = stops_1km
        features['stop_density_1km'] = stops_1km / (np.pi * 1**2)
        
        features.update(get_population_features(lat, lon, ward, df1_agg))
        features.update(get_service_features(None, service_dict, gtfs_stop_times, gtfs_trips, gtfs_stops, lat, lon))
        features.update(get_traffic_features(lat, lon, model2_daily, cell_daily_tree, cell_daily_ids))
        features.update(get_ward_features(ward, ward_features))
        features.update(get_spatial_features(lat, lon))
        features.update(get_derived_features(features))
        
        features['is_good_stop'] = 0
        
        candidates.append(features)
        attempts += 1
    
    return candidates

In [19]:
print("Generating negative examples...")
n_negatives_per_ward = N_TEST_NEG_PER_WARD if TEST_MODE else 50  # Reduce for test
if TEST_MODE:
    # Only wards from the test stops_df
    test_wards = stops_df['ward'].unique()
    ward_data_list = [(ward, stops_df[stops_df['ward'] == ward]) for ward in test_wards]
    print(f"TEST MODE: Generating negatives only for {len(ward_data_list)} wards from test stops")
else:
    ward_data_list = [(ward, stops_df[stops_df['ward'] == ward]) for ward in ward_features['ward'].unique() if len(stops_df[stops_df['ward'] == ward]) > 0]
gen_func = partial(
    generate_ward_candidates,
    n_negatives_per_ward=n_negatives_per_ward,
    G=G,
    nodes_gdf=nodes_gdf,
    stop_tree=stop_tree,
    stop_coords=stop_coords,
    df1_agg=df1_agg,
    gtfs_stop_times=gtfs_stop_times,
    gtfs_trips=gtfs_trips,
    gtfs_stops=gtfs_stops,
    service_dict=service_dict,
    model2_daily=model2_daily,
    cell_daily_tree=cell_daily_tree,
    cell_daily_ids=cell_daily_ids,
    ward_features=ward_features
)
n_cores_neg = min(n_cores, len(ward_data_list))  # Don't over-parallelize if few wards
print(f"Parallel generating {n_negatives_per_ward} negatives per ward across {len(ward_data_list)} wards...")
with Pool(n_cores_neg) as pool:
    ward_candidates = list(tqdm(
        pool.imap(gen_func, ward_data_list, chunksize=1),  # One ward per task
        total=len(ward_data_list),
        desc="Generating negatives"
    ))

negatives = [cand for ward_cands in ward_candidates for cand in ward_cands]
negatives_df = pd.DataFrame(negatives)
print(f"Generated {len(negatives_df)} negative examples")
negatives_df.head(2)

Generating negative examples...
TEST MODE: Generating negatives only for 10 wards from test stops
Parallel generating 10 negatives per ward across 10 wards...


Generating negatives: 100%|██████████| 10/10 [01:00<00:00,  6.01s/it]


Generated 100 negative examples


,stop_id,lat,lon,ward,is_existing_stop,nearest_node_degree,is_intersection,road_type,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_within_500m,pop_within_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,dominant_congestion_level,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,is_benchmark_ward,ward_service_per_capita,ward_category,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop
0,CANDIDATE_Nairobi Central Ward_0,-1.293395,36.837586,Nairobi Central Ward,False,6,True,unclassified,382.197240,336.859050,345.217551,348.956445,343.677682,5.057347,5,36,11.459156,4739,18956,6034.16,0,947.843636,0,0,0,0,NaN,0,15,17.243301,11.884734,20.101203,63.471160,93.750000,190,84.0,0.583137,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,2.396045,2.702703,4739.0,0.015658,0.0,0
1,CANDIDATE_Nairobi Central Ward_1,-1.274533,36.835489,Nairobi Central Ward,False,2,False,residential,47.368259,367.030608,370.067903,422.414569,386.504360,25.422610,12,68,21.645072,4739,18956,6034.16,0,947.843636,0,0,0,0,NaN,0,11,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,2.423777,1.449275,4739.0,0.041349,0.0,0


In [23]:
# combine and save
print("Combining datasets...")
final_df = pd.concat([stops_df, negatives_df], ignore_index=True)
# save
filename = '/home/dataopske/Desktop/jav/data/training_data/stop_features_test.csv' if TEST_MODE else 'stop_features_complete.csv'
final_df.to_csv(filename, index=False)
print(f"\n Complete! ({'TEST MODE' if TEST_MODE else 'FULL RUN'})")
print(f"Total samples: {len(final_df)}")
print(f" Existing stops: {len(stops_df)}")
print(f" Candidates: {len(negatives_df)}")
print(f" Positive labels: {final_df['is_good_stop'].sum()}")
print(f" Negative labels: {(final_df['is_good_stop'] == 0).sum()}")
print(f"\nSaved to: {filename}")
final_df.head()

Combining datasets...

 Complete! (TEST MODE)
Total samples: 200
 Existing stops: 100
 Candidates: 100
 Positive labels: 20
 Negative labels: 180

Saved to: /home/dataopske/Desktop/jav/data/training_data/stop_features_test.csv


,stop_id,lat,lon,ward,is_existing_stop,nearest_node_degree,is_intersection,road_type,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,dominant_congestion_level,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,is_benchmark_ward,ward_service_per_capita,ward_category,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop,pop_within_200m,pop_within_500m,pop_within_1km
0,0001RLW,-1.290884,36.828242,Nairobi Central Ward,True,4,True,unclassified,185.404122,173.684507,231.853573,412.893917,272.810666,101.860683,9,57,18.143664,6034.16,0,947.843636,7,14,1.75,0,102.857143,1,7,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,1.324902,1.724138,372.666667,0.021459,0.0,1,799.0,5590,27759
1,0002KOJ,-1.281230,36.822596,Nairobi Central Ward,True,3,True,residential,20.568573,5.177885,21.286282,23.097604,16.520590,8.054520,14,66,21.008452,6034.16,0,947.843636,0,0,0.00,0,NaN,0,46,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.830851,1.492537,5161.000000,0.139091,0.0,1,757.0,5161,23044
2,0003NGR,-1.274395,36.823806,Ngara Ward Ward,True,3,True,primary,58.275689,27.998313,31.421882,53.807529,37.742574,11.445298,23,64,20.371833,9887.15,0,1553.069890,0,0,0.00,0,NaN,0,26,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,25589.814342,9887.15,20.0,0.0,True,62.733293,well_served,1.523563,1.538462,7734.000000,0.050611,0.0,1,1240.0,7734,26486
3,0004ODN,-1.282769,36.825032,Nairobi Central Ward,True,3,True,tertiary,67.183239,14.009785,64.101044,81.767482,53.292770,28.698321,26,76,24.191551,6034.16,0,947.843636,0,0,0.00,0,NaN,0,54,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.959720,1.298701,4809.000000,0.043999,0.0,1,757.0,4809,25851
4,0005AMB,-1.285963,36.826048,Nairobi Central Ward,True,4,True,secondary_link,60.725155,17.426995,68.153819,103.673253,63.084689,35.391866,29,65,20.690143,6034.16,0,947.843636,0,0,0.00,0,NaN,0,65,17.684658,11.666698,20.693638,60.458835,92.241379,543,224.0,0.490429,congested,100.0,18997.268882,6034.16,20.0,0.0,True,148.372205,well_served,0.984804,1.515152,4731.000000,0.064803,0.0,1,757.0,4731,26761


In [24]:
final_df.describe()

,lat,lon,nearest_node_degree,distance_to_major_road,distance_to_nearest_stop,distance_to_2nd_nearest,distance_to_3rd_nearest,avg_spacing_3_nearest,spacing_regularity,stops_within_500m,stops_within_1km,stop_density_1km,pop_density_500m,pop_not_served_nearby,poverty_rate_weighted_pop,route_count_serving,trips_per_day,trips_per_hour_peak,trips_per_hour_offpeak,avg_headway_minutes,service_span_hours,routes_within_500m,avg_speed_daily,avg_speed_peak,avg_speed_offpeak,congestion_pct_daily,congestion_pct_peak,trip_count_daily,trip_count_peak,demand_variability_cv,ward_pct_access,ward_population,ward_pop_density,ward_poverty_rate,ward_pop_not_served,ward_service_per_capita,distance_to_cbd,coverage_efficiency_nearby,demand_supply_ratio,network_accessibility,equity_score,is_good_stop,pop_within_200m,pop_within_500m,pop_within_1km
count,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.00000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.0,89.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,200.000000,2.000000e+02,200.000000,100.000000,200.0000,200.000000
mean,-1.214042,36.816116,4.955000,328.331503,371.780846,462.952163,533.761502,456.164837,71.674210,7.58000,26.145000,8.322212,12709.955150,10913.265000,1262.672399,0.640000,0.675000,0.084375,0.0,1228.391653,0.025000,5.255000,17.826750,11.684069,20.917273,58.126479,92.524381,396.170000,160.880000,0.456085,65.365703,93043.105653,12709.955150,16.102000,36379.130000,28.162356,10.584833,7.855189,8567.704500,0.151840,8.052726e+06,0.100000,463.850000,8993.3300,36441.335000
std,0.045137,0.071033,1.894975,317.685407,500.960578,516.598933,593.343753,528.536719,99.339142,8.04095,23.738398,7.556167,17593.096395,8516.449308,1275.431101,0.992636,1.295285,0.161911,0.0,373.582363,0.156517,12.043471,0.286667,0.165457,0.391480,2.848381,0.743055,142.513655,58.013996,0.036949,25.857336,48542.140116,17593.096395,3.527351,28388.643466,41.822551,6.326222,14.918547,14268.162602,0.331558,7.001181e+06,0.300753,1093.860455,14184.4327,56937.711226
min,-1.297922,36.640396,2.000000,0.903475,0.730793,6.450994,20.499269,16.520590,0.139332,0.00000,0.000000,0.000000,3569.890000,0.000000,405.623790,0.000000,0.000000,0.000000,0.0,102.857143,0.000000,0.000000,16.977835,11.514725,20.060219,54.693287,88.888889,160.000000,66.000000,0.411452,30.372612,18997.268882,3569.890000,7.500000,0.000000,0.985044,0.395954,0.925926,0.000000,0.001561,0.000000e+00,0.000000,0.000000,0.0000,0.000000
25%,-1.254794,36.771724,3.000000,33.270699,23.509315,108.190754,152.181258,98.913195,15.804253,1.00000,6.000000,1.909859,3569.890000,412.000000,501.877524,0.000000,0.000000,0.000000,0.0,720.000000,0.000000,1.000000,17.684658,11.535665,20.693638,56.990763,92.045455,251.000000,100.000000,0.433371,51.540508,57375.096470,3569.890000,12.500000,1376.000000,4.229477,6.220680,1.847302,0.000000,0.008654,4.928666e+05,0.000000,0.000000,0.0000,1354.000000
50%,-1.204794,36.819624,6.000000,271.489867,316.364397,365.516320,409.848060,362.314546,41.966893,5.00000,19.000000,6.047888,6034.160000,11984.000000,947.843636,0.000000,0.000000,0.000000,0.0,1440.000000,0.000000,1.000000,17.733884,11.615866,20.832993,57.133067,92.968750,520.000000,208.000000,0.433371,51.540508,69612.843253,6034.160000,17.900000,39949.000000,16.862692,10.632923,2.819195,3331.250000,0.017720,9.934164e+06,0.000000,0.000000,4520.0000,18080.000000
75%,-1.183690,36.874657,6.000000,504.059405,483.091022,602.872473,676.591696,576.710495,92.206101,12.00000,40.500000,12.891550,8509.210000,19794.000000,1196.278105,1.000000,1.000000,0.125000,0.0,1440.000000,0.000000,4.000000,18.047650,11.942590,21.100180,60.458835,93.181818,520.000000,208.000000,0.490429,98.023042,136158.568704,8509.210000,17.900000,65982.000000,16.862692,14.602302,6.074522,6

In [25]:
final_df.isna().sum()  # Check for NaNs

stop_id                         0
lat                             0
lon                             0
ward                            0
is_existing_stop                0
nearest_node_degree             0
is_intersection                 0
road_type                       0
distance_to_major_road          0
distance_to_nearest_stop        0
distance_to_2nd_nearest         0
distance_to_3rd_nearest         0
avg_spacing_3_nearest           0
spacing_regularity              0
stops_within_500m               0
stops_within_1km                0
stop_density_1km                0
pop_density_500m                0
pop_not_served_nearby           0
poverty_rate_weighted_pop       0
route_count_serving             0
trips_per_day                   0
trips_per_hour_peak             0
trips_per_hour_offpeak          0
avg_headway_minutes           111
service_span_hours              0
routes_within_500m              0
avg_speed_daily                 0
avg_speed_peak                  0
avg_speed_offp